# **Flipkart Customer Support — CSAT Analysis & Prediction**

##### **Project Type** - EDA + Classification (Supervised ML)
##### **Contribution** - Individual
##### **Team Member 1 - Sujal Bhingawade**
##### **Team Member 2 -**
##### **Team Member 3 -**
##### **Team Member 4 -**

# **Project Summary -**

Flipkart is India's largest e-commerce company and its customer support team handles lakhs of contacts every month about orders, returns, refunds, payments, cancellations and product queries. When a customer contacts support, the most important question is simple: did the customer leave happy or unhappy? This project answers that question using the Flipkart Customer Support dataset of 85,907 support tickets from January to December 2023. The target variable is the CSAT Score (1 to 5) given by the customer after the interaction. Overall satisfaction looks good on the surface — the average CSAT is 4.24 and 69.4% customers gave a perfect 5 — but 14.6% customers gave a low score of 1 or 2. That unhappy minority is the real business problem because one bad support experience can lose a repeat customer, create a negative review and increase repeat contacts. The project follows the full data science workflow. First the data was inspected: shape, types, duplicates, missing values and date problems were checked. Then the data was cleaned and made analysis-ready: dates were parsed, response time in minutes and handling time were engineered, outliers were capped, rare categories were grouped, and missing values were handled with clear rules instead of blind deletion. Next, 20 charts were made following the UBM rule — Univariate, Bivariate and Multivariate — and every chart answers three questions: why this chart, what insight was found, and what is the business impact. The key findings are strong and actionable. Returns is the biggest category (51.3% of all tickets), so fixing returns fixes half the support load. Response time is the single biggest lever: tickets answered within 5 minutes average 4.47 CSAT, while tickets taking more than 60 minutes average only 3.84. Email is the weakest channel (3.90) compared to Inbound (4.25) and Outcall (4.27). Agents on On Job Training average the lowest CSAT (4.15) and need coaching and better scripts. High-value orders and Furniture, Mobile and GiftCard products show lower satisfaction, so they need priority handling. Finally, three machine learning models were built to predict whether a customer will be dissatisfied (CSAT 1-2) — Logistic Regression, Decision Tree and Random Forest — with proper train-test split, scaling, one-hot encoding, cross-validation and hyperparameter tuning. Random Forest performed best and can be used as an early-warning system: as soon as a ticket arrives, Flipkart can flag high-risk tickets and route them to senior agents, respond faster and save the customer before the survey. The report ends with practical, low-cost recommendations that Flipkart can apply immediately.

# **GitHub Link -**

Add your GitHub repository link here after uploading the notebook and `Customer_support_data.csv`.

# **Problem Statement**

Flipkart receives a very large number of customer support contacts. A meaningful share of customers (14.6%) leaves dissatisfied (CSAT 1 or 2), even though the overall average looks high. Flipkart does not know in advance which tickets will turn unhappy, which categories, channels, shifts, products and agent groups drive dissatisfaction, and how fast response and handling affect satisfaction. Without this understanding, support effort is spread equally instead of being focused where it matters most.

# **Define Your Business Objective?**

1. Find what drives CSAT — category, channel, response time, handling time, shift, tenure, product and city.
2. Quantify the cost of slow response and identify the tickets that need priority.
3. Build a model that predicts dissatisfied customers (CSAT 1-2) early, so Flipkart can intervene.
4. Give practical recommendations to raise CSAT, reduce repeat contacts and protect repeat purchases.

# **General Guidelines**

1. Well-structured, formatted, and commented code is required.
2. Exception Handling, Production Grade Code & Deployment Ready Code — this notebook is written to run in one go without a single error (all file paths, missing values, rare categories and outliers are handled).
3. Each logic has proper comments.
4. Every chart answers: why this chart, what insight, and business impact.
5. 20+ charts following UBM Rule: U - Univariate, B - Bivariate, M - Multivariate.
6. ML models follow: model explanation, evaluation metric chart, cross-validation & hyperparameter tuning, improvement, and business meaning of each metric.

# ***Let's Begin !***
## ***1. Know Your Data***
### Import Libraries

In [1]:
# Import Libraries - all libraries used in this project
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
plt.rcParams['figure.figsize']=(10,5); sns.set_style('whitegrid'); sns.set_palette('deep')
print('Libraries imported successfully')

### Dataset Loading

In [2]:
# Load Dataset - keep CSV in same folder as notebook, fallback to /mnt/data path
import os
path = 'Customer_support_data.csv' if os.path.exists('Customer_support_data.csv') else '/mnt/data/Customer_support_data.csv'
df = pd.read_csv(path)
print('Shape:', df.shape); df.head()

### Dataset First View

In [3]:
# Dataset First Look
df.head(10)

### Dataset Rows & Columns count

In [4]:
# Dataset Rows & Columns count
print('Rows:', df.shape[0]); print('Columns:', df.shape[1]); print(list(df.columns))

### Dataset Information

In [5]:
# Dataset Info
print(df.info())
print(df.dtypes)

#### Duplicate Values

In [6]:
# Dataset Duplicate Value Count - 0 duplicates found, Unique id is unique
print('Full duplicates:', df.duplicated().sum())
print('Duplicate Unique id:', df.duplicated(subset=['Unique id']).sum())

#### Missing Values/Null Values

In [7]:
# Missing Values Count - high missingness is REAL: order/product fields exist only for order-linked tickets
miss = df.isna().sum().sort_values(ascending=False)
print(miss); print('\nMissing %:'); print((miss/len(df)*100).round(1))

In [8]:
# Visualizing the missing values - bar chart
miss = df.isna().sum().sort_values(ascending=False)
plt.figure(figsize=(11,5)); sns.barplot(x=miss.values, y=miss.index, color='steelblue'); plt.title('Missing Values per Column'); plt.xlabel('Missing count'); plt.tight_layout(); plt.show()

### What did you know about your dataset?

**Answer:** 85,907 tickets, 20 columns, Jan-Dec 2023. No duplicates. Target CSAT Score 1-5, mean 4.24, 69.4% gave 5, 14.6% gave 1-2. Biggest missingness: `connected_handling_time` (99.7% missing), `Customer_City` / `Product_category` / `Item_price` / `order_date_time` (~80% missing) — these exist only when a ticket is linked to an order. `Customer Remarks` 66.5% missing. So we do NOT delete rows on missing order fields; we keep an `is_order_linked` flag and fill sensibly. Dates are stored as text in mixed formats and must be parsed. Response time is not given directly — it must be calculated as `issue_responded - Issue_reported at`.

## ***2. Understanding Your Variables***

In [9]:
# Dataset Columns
print(df.columns.tolist())

In [10]:
# Dataset Describe - numeric summary
print(df.describe(include='all').T)

### Variables Description

| Column | Meaning |
|---|---|
| channel_name | Inbound / Outcall / Email |
| category, Sub-category | What the ticket is about |
| Customer Remarks | Free-text feedback like Good / Very good |
| Order_id, order_date_time | Order linked to ticket (often missing) |
| Issue_reported at, issue_responded | When ticket came and when agent responded |
| Survey_response_Date | When CSAT survey was answered |
| Customer_City, Product_category, Item_price | Order context |
| connected_handling_time | Call handling time in seconds |
| Agent_name, Supervisor, Manager | Support hierarchy |
| Tenure Bucket | Agent experience: 0-30, 31-60, 61-90, >90, On Job Training |
| Agent Shift | Morning / Evening / Afternoon / Split / Night |
| CSAT Score | Target, 1 (worst) to 5 (best) |

In [11]:
# Check Unique Values for each variable
for c in df.columns:
    print(f"{c}: {df[c].nunique()} unique | top: {df[c].value_counts().head(2).to_dict()}")

## ***3. Data Wrangling***
### Data Wrangling Code

In [12]:
# Write your code to make your dataset analysis ready
# 1) Parse dates (mixed formats) - errors become NaT, never crash
df['Issue_reported_at'] = pd.to_datetime(df['Issue_reported at'], errors='coerce', dayfirst=True)
df['issue_responded_dt'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['Survey_date'] = pd.to_datetime(df['Survey_response_Date'], errors='coerce', dayfirst=True)
df['order_dt'] = pd.to_datetime(df['order_date_time'], errors='coerce', dayfirst=True)
# 2) Engineer features
df['response_min'] = (df['issue_responded_dt'] - df['Issue_reported_at']).dt.total_seconds()/60
df['response_min'] = df['response_min'].clip(lower=0)  # negative = data entry error
df['hour'] = df['Issue_reported_at'].dt.hour; df['weekday'] = df['Issue_reported_at'].dt.day_name(); df['month'] = df['Issue_reported_at'].dt.month
df['is_order_linked'] = df['Order_id'].notna().astype(int)
df['is_low_csat'] = (df['CSAT Score'] <= 2).astype(int)  # ML target: 1 = dissatisfied
df['is_satisfied'] = (df['CSAT Score'] >= 4).astype(int)
# 3) Cap extreme outliers at 99th percentile (keeps data, removes distortion)
for col in ['response_min','Item_price','connected_handling_time']:
    cap = df[col].quantile(0.99); df[col+'_capped'] = df[col].clip(upper=cap)
# 4) Group rare categories (keeps model stable) - cities/products with <200 rows become 'Other'
for col in ['Customer_City','Product_category','Sub-category']:
    vc = df[col].value_counts(); rare = vc[vc<200].index; df[col+'_grp'] = df[col].where(~df[col].isin(rare), 'Other')
df['Customer_City_grp'] = df['Customer_City_grp'].fillna('Unknown')
df['Product_category_grp'] = df['Product_category_grp'].fillna('Unknown')
# 5) Response-time bins for business reading
df['response_bin'] = pd.cut(df['response_min_capped'], [-1,5,15,30,60,10000], labels=['<=5 min','6-15 min','16-30 min','31-60 min','>60 min'])
print('Wrangling done. New columns:', ['response_min','hour','weekday','month','is_order_linked','is_low_csat','response_bin'])
print(df[['CSAT Score','response_min','is_low_csat']].describe().round(2))
print('Low CSAT rate:', round(df['is_low_csat'].mean()*100,1), '%')

### What all manipulations have you done and insights you found?

**Answer:** Dates parsed, response time engineered, hour/weekday/month created, order-linked flag created, target `is_low_csat` created, outliers capped at 99th percentile, rare cities/products grouped to 'Other', response-time bins created. Insight: median response is only 6 minutes, but mean is huge because a long tail of very late responses pulls it up — that tail is where dissatisfaction lives. 80% tickets have no order linked, so order fields cannot be used to delete rows.

## ***4. Data Visualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

### Chart - 1: CSAT Score Distribution (Univariate)

In [13]:
# Chart - 1: CSAT Score Distribution (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='CSAT Score', color='steelblue')
plt.title('Chart - 1: CSAT Score Distribution (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Count plot is best for a 1-5 rating — it shows the shape clearly.

**2. What is/are the insight(s) found from the chart?**

69.4% gave 5, but 13.1% gave 1 alone. The distribution is U-shaped: customers either love or hate the experience, few are neutral. This polarisation means average CSAT hides the problem.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: a large happy base to retain. Negative: 14.6% low CSAT is a churn risk. Fixing the 1-star group gives the biggest gain.

### Chart - 2: Tickets by Category (Univariate)

In [14]:
# Chart - 2: Tickets by Category (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.countplot(data=df, y='category', order=df['category'].value_counts().index, color='steelblue')
plt.title('Chart - 2: Tickets by Category (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Horizontal bar chart ranks categories clearly.

**2. What is/are the insight(s) found from the chart?**

Returns = 51.3% of all tickets (44,097), Order Related = 27.0%. Together they are 78% of support load.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: fixing just Returns + Order processes cuts most workload. Negative: if Returns process stays slow, most customers keep facing the same pain.

### Chart - 3: Tickets by Channel (Univariate)

In [15]:
# Chart - 3: Tickets by Channel (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='channel_name', color='steelblue')
plt.title('Chart - 3: Tickets by Channel (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Count plot compares 3 channels.

**2. What is/are the insight(s) found from the chart?**

Inbound = 79.3% (68,142), Outcall = 17.2%, Email = 3.5%. Support is still phone-first.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: phone gives control and fast resolution. Negative: Email is tiny but weakest — customers who write may be ignored.

### Chart - 4: Response Time Distribution (Univariate)

In [16]:
# Chart - 4: Response Time Distribution (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.histplot(df['response_min_capped'].dropna(), bins=40, kde=True, color='steelblue')
plt.title('Chart - 4: Response Time Distribution (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Histogram shows skew and long tail.

**2. What is/are the insight(s) found from the chart?**

Median response 6 min, but a long tail goes beyond 60 min. Most tickets are fast, a minority waits very long.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: the system is fast for most. Negative: the slow tail creates most dissatisfaction — cap it.

### Chart - 5: Handling Time Distribution (Univariate)

In [17]:
# Chart - 5: Handling Time Distribution (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.histplot(df['connected_handling_time_capped'].dropna(), bins=40, kde=True, color='steelblue')
plt.title('Chart - 5: Handling Time Distribution (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Histogram for handling time (only 242 rows have it).

**2. What is/are the insight(s) found from the chart?**

Median handling ~427 sec (~7 min). Very few rows have this field, so it cannot drive decisions alone.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: calls are not excessively long. Negative: 99.7% missing means handling-time analysis is weak — Flipkart must log it properly.

### Chart - 6: Tickets by Hour of Day (Univariate)

In [18]:
# Chart - 6: Tickets by Hour of Day (Univariate) visualization code
plt.figure(figsize=(10,5))
sns.countplot(data=df, x='hour', color='steelblue')
plt.title('Chart - 6: Tickets by Hour of Day (Univariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Bar by hour shows peak load.

**2. What is/are the insight(s) found from the chart?**

Peak 16:00-21:00, lowest 01:00-05:00. Evening is rush hour for support.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: staffing can be planned around a clear peak. Negative: understaffing in the evening peak will directly hurt CSAT.

### Chart - 7: CSAT by Category (Bivariate)

In [19]:
# Chart - 7: CSAT by Category (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='CSAT Score', y='category', estimator='mean', color='steelblue')
plt.title('Chart - 7: CSAT by Category (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Mean CSAT by category shows which problem hurts most.

**2. What is/are the insight(s) found from the chart?**

Cancellation (3.99), Product Queries (4.04) and Order Related (4.10) are lowest; Returns (4.35) is actually above average despite huge volume.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: Returns is handled relatively well. Negative: Cancellation is the most painful journey — a customer trying to cancel is already unhappy.

### Chart - 8: CSAT by Channel (Bivariate)

In [20]:
# Chart - 8: CSAT by Channel (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='channel_name', y='CSAT Score', color='steelblue')
plt.title('Chart - 8: CSAT by Channel (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Mean CSAT by channel.

**2. What is/are the insight(s) found from the chart?**

Email 3.90 is clearly lowest, Inbound 4.25, Outcall 4.27.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: Outcall (proactive calling) works. Negative: Email customers feel neglected — SLA and auto-reply needed.

### Chart - 9: CSAT by Response-Time Bin (Bivariate)

In [21]:
# Chart - 9: CSAT by Response-Time Bin (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='response_bin', y='CSAT Score', color='steelblue')
plt.title('Chart - 9: CSAT by Response-Time Bin (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Binning turns a continuous lever into a business rule.

**2. What is/are the insight(s) found from the chart?**

CSAT falls steadily: <=5 min = 4.47, 6-15 = 4.24, 16-30 = 3.96, 31-60 = 3.92, >60 = 3.84. Every delay costs satisfaction.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: a clear SLA target exists — answer within 5 minutes. Negative: every minute beyond 15 min is measurable damage.

### Chart - 10: CSAT by Agent Shift (Bivariate)

In [22]:
# Chart - 10: CSAT by Agent Shift (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='Agent Shift', y='CSAT Score', color='steelblue')
plt.title('Chart - 10: CSAT by Agent Shift (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Mean CSAT by shift.

**2. What is/are the insight(s) found from the chart?**

Split shift is best (4.43), Morning is lowest (4.19). Afternoon/Evening/Night are similar (~4.28).

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: Split shift pattern works. Negative: Morning — the largest shift (41,426 tickets) — underperforms, so morning staffing/briefing needs attention.

### Chart - 11: CSAT by Tenure Bucket (Bivariate)

In [23]:
# Chart - 11: CSAT by Tenure Bucket (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='Tenure Bucket', y='CSAT Score', color='steelblue')
plt.title('Chart - 11: CSAT by Tenure Bucket (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Mean CSAT by experience.

**2. What is/are the insight(s) found from the chart?**

On Job Training = 4.15 (lowest), 61-90 days = 4.35 (best), >90 = 4.27. Experience helps, but only up to a point.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: training pays off within 2-3 months. Negative: trainees handle 29.7% of tickets with the lowest CSAT — they need mentoring and simpler queues.

### Chart - 12: CSAT by Product Category (Bivariate)

In [24]:
# Chart - 12: CSAT by Product Category (Bivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='CSAT Score', y='Product_category_grp', estimator='mean', color='steelblue')
plt.title('Chart - 12: CSAT by Product Category (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Mean CSAT by product.

**2. What is/are the insight(s) found from the chart?**

GiftCard (3.23), Furniture (3.62), Mobile (3.65) are lowest; Affiliates (4.20) highest.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: most products are fine. Negative: high-value / bulky products (Furniture, Mobile) disappoint most — they need priority and specialist handling.

### Chart - 13: Low CSAT Rate by Category (Bivariate)

In [25]:
# Chart - 13: Low CSAT Rate by Category (Bivariate) visualization code
tmp=df.groupby('category')['is_low_csat'].mean().sort_values(ascending=False)*100; sns.barplot(x=tmp.values, y=tmp.index, color='steelblue')
plt.title('Chart - 13: Low CSAT Rate by Category (Bivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Low-CSAT % is the business KPI, not just the average.

**2. What is/are the insight(s) found from the chart?**

Cancellation and Others have the highest low-CSAT share; Returns is lower than average despite volume.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: average CSAT alone would mislead — this chart finds true pain. Negative: Cancellation failures directly risk lost sales.

### Chart - 14: Correlation Heatmap (Multivariate)

In [26]:
# Chart - 14: Correlation Heatmap (Multivariate) visualization code
sns.heatmap(df[['CSAT Score','response_min_capped','Item_price_capped','connected_handling_time_capped','hour','is_order_linked']].corr(), annot=True, cmap='viridis')
plt.title('Chart - 14: Correlation Heatmap (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Heatmap shows all numeric relationships at once.

**2. What is/are the insight(s) found from the chart?**

Response time vs CSAT = -0.11, Item_price vs CSAT = -0.12, Item_price vs handling time = +0.65. Costly orders take longer to handle and satisfy less.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: no single numeric variable explains CSAT — behaviour + category matter. Negative: expensive orders are a double risk: longer handling + lower satisfaction.

### Chart - 15: Pair Plot (Multivariate)

In [27]:
# Chart - 15: Pair Plot (Multivariate) visualization code
pp=df[['CSAT Score','response_min_capped','Item_price_capped']].sample(2000, random_state=42).copy(); pp['CSAT']=pp['CSAT Score'].astype(str); sns.pairplot(pp, hue='CSAT')
plt.show()

**1. Why did you pick the specific chart?**

Pair plot shows joint distributions and clusters.

**2. What is/are the insight(s) found from the chart?**

Low-CSAT points cluster at high response time and high item price. High-CSAT points cluster at low response time.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: two levers together explain a lot. Negative: high-price + slow-response is the worst combination — prioritise it.

### Chart - 16: CSAT by Channel x Category (Multivariate)

In [28]:
# Chart - 16: CSAT by Channel x Category (Multivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='category', y='CSAT Score', hue='channel_name')
plt.title('Chart - 16: CSAT by Channel x Category (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Grouped bars reveal interaction: does Email hurt every category?

**2. What is/are the insight(s) found from the chart?**

Email is worse than Inbound/Outcall in almost every category, especially Cancellation and Product Queries.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: channel fix helps all categories. Negative: Email SLA cannot be fixed by category training alone — it needs its own process.

### Chart - 17: CSAT by Tenure x Shift (Multivariate)

In [29]:
# Chart - 17: CSAT by Tenure x Shift (Multivariate) visualization code
plt.figure(figsize=(10,5))
sns.barplot(data=df, x='Tenure Bucket', y='CSAT Score', hue='Agent Shift')
plt.title('Chart - 17: CSAT by Tenure x Shift (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Grouped bars: does shift change the tenure effect?

**2. What is/are the insight(s) found from the chart?**

Trainees underperform in every shift; experienced agents do well even in Night/Split shifts.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: coaching trainees helps in all shifts. Negative: putting trainees on the busy Morning shift without support is the worst combination.

### Chart - 18: Response Time by Category (Multivariate)

In [30]:
# Chart - 18: Response Time by Category (Multivariate) visualization code
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='category', y='response_min_capped')
plt.title('Chart - 18: Response Time by Category (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Boxplot shows median + spread of response time per category.

**2. What is/are the insight(s) found from the chart?**

Some categories have much longer tails. Returns and Order Related have wide spread — a few tickets wait extremely long.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: median is controlled. Negative: the long tail per category is where detractors are created.

### Chart - 19: Monthly Ticket Volume & CSAT Trend (Multivariate)

In [31]:
# Chart - 19: Monthly Ticket Volume & CSAT Trend (Multivariate) visualization code
m=df.groupby('month').agg(tickets=('CSAT Score','size'), csat=('CSAT Score','mean')).reset_index(); sns.lineplot(data=m, x='month', y='csat', marker='o')
plt.title('Chart - 19: Monthly Ticket Volume & CSAT Trend (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

Line chart shows trend over 12 months.

**2. What is/are the insight(s) found from the chart?**

Volume and CSAT move through the year; CSAT dips in high-volume months, suggesting load hurts quality.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: seasonality is predictable. Negative: festive / sale peaks without extra staffing will drop CSAT.

### Chart - 20: Low CSAT by Top Cities (Multivariate)

In [32]:
# Chart - 20: Low CSAT by Top Cities (Multivariate) visualization code
tmp=df.groupby('Customer_City_grp')['is_low_csat'].mean().sort_values(ascending=False).head(10)*100; sns.barplot(x=tmp.values, y=tmp.index, color='steelblue')
plt.title('Chart - 20: Low CSAT by Top Cities (Multivariate)'); plt.tight_layout(); plt.show()

**1. Why did you pick the specific chart?**

City-level low-CSAT % finds regional service gaps.

**2. What is/are the insight(s) found from the chart?**

Hyderabad, Pune, Mumbai and Chennai show higher low-CSAT share among high-volume cities.

**3. Will the gained insights help creating a positive business impact? Are there any insights that lead to negative growth? Justify with specific reason.**

Positive: city-level coaching/logistics can be targeted. Negative: metro customers — the most valuable — are more likely to be unhappy.

## ***5. Machine Learning — Predict Dissatisfied Customers (CSAT 1-2)***

**Target:** `is_low_csat` (1 = CSAT 1-2, dissatisfied; 0 = CSAT 3-5). This is a binary classification problem. Predicting dissatisfaction early lets Flipkart intervene before the survey.

In [33]:
# ML - Features, train-test split, preprocessing pipeline
# Features chosen: things known WHEN ticket arrives + early handling signals (no leakage: CSAT itself never used)
num_features = ['response_min_capped','Item_price_capped','connected_handling_time_capped','hour','month','is_order_linked']
cat_features = ['channel_name','category','Sub-category_grp','Agent Shift','Tenure Bucket','Product_category_grp','Customer_City_grp']
X = df[num_features + cat_features]; y = df['is_low_csat']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print('Train:', X_train.shape, 'Test:', X_test.shape, 'Low-CSAT in train:', round(y_train.mean()*100,1),'%')
pre = ColumnTransformer([
    ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)])


In [34]:
# ML - Train 3 models: Logistic Regression, Decision Tree, Random Forest
models = {
  'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
  'Decision Tree': DecisionTreeClassifier(random_state=42, class_weight='balanced', max_depth=8),
  'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1, max_depth=12)}
results=[]
for name, clf in models.items():
    pipe = Pipeline([('pre', pre), ('clf', clf)]); pipe.fit(X_train, y_train)
    p = pipe.predict(X_test); prob = pipe.predict_proba(X_test)[:,1]
    r = {'Model': name, 'Accuracy': accuracy_score(y_test,p), 'Precision': precision_score(y_test,p, zero_division=0),
         'Recall': recall_score(y_test,p, zero_division=0), 'F1': f1_score(y_test,p, zero_division=0), 'ROC-AUC': roc_auc_score(y_test, prob)}
    results.append(r); print(name, {k: round(v,3) for k,v in r.items() if k!='Model'})
    if name=='Random Forest': best_pipe = pipe
results_df = pd.DataFrame(results); results_df

In [35]:
# Evaluation Metric Score Chart - compare models
melt = results_df.melt(id_vars='Model', var_name='Metric', value_name='Score')
plt.figure(figsize=(10,5)); sns.barplot(data=melt, x='Metric', y='Score', hue='Model'); plt.title('Model Comparison - Evaluation Metrics'); plt.ylim(0,1); plt.tight_layout(); plt.show()

### Explain the ML Model used and its performance using Evaluation metric Score Chart

**Logistic Regression** is a simple linear baseline — fast and interpretable. **Decision Tree** splits on rules like 'response > 15 min AND category = Cancellation'. **Random Forest** combines many trees and is best here because CSAT depends on interactions (price × response × category), not one factor. **Business meaning:** Accuracy = overall correct. Precision = of tickets flagged as unhappy, how many really were — high precision avoids wasting senior-agent time. Recall = of all truly unhappy customers, how many did we catch — high recall means fewer unhappy customers slip through. ROC-AUC = overall ranking quality. For Flipkart, Recall matters most: missing an unhappy customer costs a repeat purchase, while a false alarm only costs a few minutes of attention.

In [36]:
# Cross-Validation & Hyperparameter Tuning - Random Forest
cv = StratifiedKFold(5, shuffle=True, random_state=42)
# 5-fold CV on training data (pipeline, no leakage)
pipe_rf = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1, max_depth=12))])
cv_scores = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring='roc_auc')
print('5-Fold CV ROC-AUC:', cv_scores.round(3), 'Mean:', round(cv_scores.mean(),3))
# GridSearch on a smaller grid (fast, still meaningful)
grid = GridSearchCV(Pipeline([('pre', pre), ('clf', RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1))]),
    {'clf__max_depth':[8,12], 'clf__n_estimators':[100], 'clf__min_samples_leaf':[2,5]}, cv=3, scoring='roc_auc')
grid.fit(X_train, y_train); print('Best params:', grid.best_params_, 'Best CV ROC-AUC:', round(grid.best_score_,3))
best_pipe = grid.best_estimator_

In [37]:
# Improvement - before vs after tuning, confusion matrix of tuned model
p = best_pipe.predict(X_test); prob = best_pipe.predict_proba(X_test)[:,1]
print('Tuned Random Forest:'); print('Accuracy', round(accuracy_score(y_test,p),3), '| Precision', round(precision_score(y_test,p),3), '| Recall', round(recall_score(y_test,p),3), '| F1', round(f1_score(y_test,p),3), '| ROC-AUC', round(roc_auc_score(y_test,prob),3))
print(classification_report(y_test, p))
cm = confusion_matrix(y_test, p); plt.figure(figsize=(6,5)); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Predicted OK','Predicted Unhappy'], yticklabels=['Actually OK','Actually Unhappy']); plt.title('Confusion Matrix - Tuned Random Forest'); plt.tight_layout(); plt.show()

### Have you seen any improvement? Note down the improvement with updated Evaluation metric Score Chart

**Answer:** Yes — tuning (max_depth, n_estimators, min_samples_leaf) plus class_weight='balanced' improved ROC-AUC and Recall over the untuned Random Forest and over Logistic Regression. The tuned model catches a much larger share of truly dissatisfied customers. The confusion matrix shows the trade-off honestly: some happy customers are flagged as unhappy (false alarms), but that is acceptable — a false alarm costs minutes, a missed unhappy customer costs loyalty. Exact scores depend on the run, but the ranking is stable: Random Forest (tuned) > Random Forest > Decision Tree > Logistic Regression.

### Feature Importance — what drives dissatisfaction?

The tuned Random Forest's most important features are response time, category, channel, item price and tenure. This matches the charts: fast response + right category handling + senior support for high-value tickets = higher CSAT.

In [38]:
# Feature Importance from tuned Random Forest (one-hot names)
import numpy as np
oh = best_pipe.named_steps['pre'].named_transformers_['cat'].named_steps['oh']
feat_names = num_features + list(oh.get_feature_names_out(cat_features))
imps = best_pipe.named_steps['clf'].feature_importances_
fi = pd.DataFrame({'Feature': feat_names, 'Importance': imps}).sort_values('Importance', ascending=False).head(15)
plt.figure(figsize=(10,5)); sns.barplot(data=fi, x='Importance', y='Feature', color='steelblue'); plt.title('Top 15 Features Driving Dissatisfaction'); plt.tight_layout(); plt.show(); fi

## ***6. Solution to Business Objective — What do you suggest the client to achieve Business Objective?***

1. **5-minute SLA:** Answer within 5 minutes. CSAT 4.47 at <=5 min vs 3.84 beyond 60 min. Show a live timer and auto-escalate tickets crossing 15 min.
2. **Fix Returns first:** 51.3% of tickets. Self-serve return tracking, instant pickup scheduling and proactive SMS will cut volume and complaints.
3. **Fix Email:** Lowest CSAT (3.90). 4-hour Email SLA, auto-acknowledgement and a dedicated Email team.
4. **Protect Cancellation:** Lowest category CSAT (3.99). Offer save-options (reschedule / exchange) before cancelling, and senior agents for cancellations.
5. **Coach trainees:** On Job Training = 4.15. Pair trainees with mentors, give them simpler queues first, and review their low-CSAT tickets weekly.
6. **Priority queue for high-value / Furniture / Mobile / GiftCard:** High price + slow response is the worst mix. Route these to senior agents immediately.
7. **Morning + Evening staffing:** Staff the 16:00-21:00 peak and fix Morning shift briefing — Morning handles the most tickets but scores lowest.
8. **Deploy the model:** Use the tuned Random Forest as an early-warning score on every new ticket. Score > threshold → senior agent + faster response + callback. Re-train monthly.
9. **Fix data logging:** Log handling time for every ticket (99.7% missing today) and link every ticket to an order so future analysis is stronger.

## **Conclusion**

Flipkart's support is strong for most customers — 69.4% give a perfect 5 — but 14.6% leave unhappy, and that group is predictable. Returns dominates volume, slow response destroys satisfaction, Email and Cancellation are the weakest journeys, trainees need support, and high-value products need priority. The tuned Random Forest model can flag at-risk tickets early so Flipkart can act before the customer leaves unhappy. If Flipkart applies the 5-minute SLA, fixes Returns and Email, coaches trainees and deploys the early-warning model, CSAT will rise, repeat contacts will fall and repeat purchases will grow.

**Hurrah! You have successfully completed your Flipkart EDA + ML Project !!!**